In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import gc
import json
import re
import numpy as np
from tqdm import tqdm
import evaluate
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
C:\Users\Legion\AppData\Local\Temp\ipykernel_6376\3564461060.py:11: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA Available: True
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
BASE_MODEL_DIR = "base_llama3_model"
ADAPTER_DIR = "adapters/SROIE_Adapter"
TEST_FILE = "phase1b_sroie_test_gold.jsonl"

In [3]:
print("\nLoading models for SROIE inference...")

if not os.path.exists(BASE_MODEL_DIR):
    raise FileNotFoundError(f"Local base model not found at '{BASE_MODEL_DIR}'.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_DIR, 
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

if not os.path.exists(ADAPTER_DIR):
    raise FileNotFoundError(f"SROIE Adapter not found at '{ADAPTER_DIR}'.")

print(f"Loading SROIE LoRA Adapter from: {ADAPTER_DIR}...")
model.load_adapter(ADAPTER_DIR)
FastLanguageModel.for_inference(model)


Loading models for SROIE inference...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading SROIE LoRA Adapter from: adapters/SROIE_Adapter...


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3072, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=307

In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an intelligent AI invoice extraction agent. Analyze the raw OCR text from the scanned document and extract the key billing entities.
CRITICAL RULES:
1. Extract ONLY these exact four keys: "merchant", "date", "address", and "total". Do NOT add any extra keys like cash or change.
2. Extract the values EXACTLY as they appear in the raw OCR text. Do NOT reformat dates, times, or numbers.
3. ALL values in the JSON must be formatted as strings (e.g., "7.50" instead of 7.50).
4. If a field is missing, output null.

Format your output EXACTLY like this:
Thought: [Explain your reasoning step-by-step]
JSON: 
{{
  "merchant": "...",
  "date": "...",
  "address": "...",
  "total": "..."
}}

### Input:
{}

### Response:
{}"""

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [5]:
print("Model fully loaded and ready for invoice extraction!")

print(f"\nLoading test data from {TEST_FILE}...")
test_samples = []
with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        test_samples.append(json.loads(line))

def parse_sroie_json(text):
    try:
        if "JSON:" in text:
            json_block = text.split("JSON:")[1].strip()
        else:
            json_block = text.strip()
            
        json_block = re.sub(r"```json\s*", "", json_block)
        json_block = re.sub(r"```\s*$", "", json_block)
        
        return json.loads(json_block)
    except Exception:
        return {"merchant": "[MISSING]", "date": "[MISSING]", "total": "[MISSING]", "address": "[MISSING]"}

results = []

Model fully loaded and ready for invoice extraction!

Loading test data from phase1b_sroie_test_gold.jsonl...


In [6]:
print("\nAgent is evaluating unseen invoices...")

for sample in tqdm(test_samples, desc="Evaluating Invoices"): 
    unseen_invoice = sample["input"]
    gold_full_text = sample["output"] 
    
    inputs = tokenizer([alpaca_prompt.format(unseen_invoice, "Thought: ")], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 800,      
        temperature = 0.1,          
        top_p = 0.9,                
        use_cache = True,
        repetition_penalty = 1.1,
        eos_token_id = terminators, 
    )
    
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    try:
        ai_full_text = generated_text.split("### Response:\n")[1].strip()
    except IndexError:
        ai_full_text = generated_text.strip()
        
    results.append({
        "Generated_JSON": parse_sroie_json(ai_full_text), 
        "Gold_JSON": parse_sroie_json(gold_full_text)
    })


Agent is evaluating unseen invoices...


Evaluating Invoices: 100%|██████████| 126/126 [1:02:16<00:00, 29.65s/it]


In [7]:
print("\nLoading BERTScore Model...")
bertscore = evaluate.load("bertscore")

def get_bert_f1(preds, refs):

    clean_preds = [str(p).strip() if str(p).strip() else "[MISSING]" for p in preds]
    clean_refs =  [str(r).strip() if str(r).strip() else "[MISSING]" for r in refs]
    
    scores = bertscore.compute(predictions=clean_preds, references=clean_refs, lang="en")
    return np.mean(scores['f1']) * 100

pred_merchants, pred_dates, pred_totals, pred_addresses = [], [], [], []
ref_merchants, ref_dates, ref_totals, ref_addresses = [], [], [], []


Loading BERTScore Model...


In [8]:
for res in results:
    ai = res["Generated_JSON"]
    gold = res["Gold_JSON"]
    
    pred_merchants.append(ai.get("merchant", "[MISSING]"))
    pred_dates.append(ai.get("date", "[MISSING]"))
    pred_totals.append(ai.get("total", "[MISSING]"))
    pred_addresses.append(ai.get("address", "[MISSING]"))
    
    ref_merchants.append(gold.get("merchant", "[MISSING]"))
    ref_dates.append(gold.get("date", "[MISSING]"))
    ref_totals.append(gold.get("total", "[MISSING]"))
    ref_addresses.append(gold.get("address", "[MISSING]"))

f1_merchants = get_bert_f1(pred_merchants, ref_merchants)
f1_dates = get_bert_f1(pred_dates, ref_dates)
f1_totals = get_bert_f1(pred_totals, ref_totals)
f1_addresses = get_bert_f1(pred_addresses, ref_addresses)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
print("\n" + "="*50)
print("SROIE EXTRACTION ACCURACY (BERTScore F1)")
print("="*50)
print(f"[MERCHANT] Accuracy: {f1_merchants:.2f}%")
print(f"[DATE]     Accuracy: {f1_dates:.2f}%")
print(f"[TOTAL]    Accuracy: {f1_totals:.2f}%")
print(f"[ADDRESS]  Accuracy: {f1_addresses:.2f}%")
print("-" * 50)
overall_avg = (f1_merchants + f1_dates + f1_totals + f1_addresses) / 4
print(f"OVERALL PIPELINE SCORE:  {overall_avg:.2f}%")
print("="*50)


SROIE EXTRACTION ACCURACY (BERTScore F1)
[MERCHANT] Accuracy: 86.22%
[DATE]     Accuracy: 84.06%
[TOTAL]    Accuracy: 89.20%
[ADDRESS]  Accuracy: 83.18%
--------------------------------------------------
OVERALL PIPELINE SCORE:  85.67%


In [10]:
def evaluate_custom_invoice(ocr_text, title):
    print(f"\nEvaluating: {title}...\n" + "="*40)
    
    inputs = tokenizer([alpaca_prompt.format(ocr_text, "Thought: ")], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 800,       
        temperature = 0.1,          
        top_p = 0.9,                
        use_cache = True,
        repetition_penalty = 1.1,   
        eos_token_id = terminators, 
    )

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    try:
        response_only = generated_text.split("### Response:\n")[1].strip()
        
        json_start = response_only.find('{')
        json_end = response_only.rfind('}') + 1 
        
        if json_start != -1 and json_end != 0:

            thoughts = response_only[:json_start].strip()
            clean_json = response_only[json_start:json_end]
            
            print(thoughts)
            print("\nJSON:")
            print(clean_json)
        else:
            print(response_only) # Fallback if no JSON found
            
    except IndexError:
        print(generated_text.strip())

In [11]:
# Unseen Custom Invoice
unseen_mr_diy = "MR D.I.Y. (M) SDN BHD [83, 169, 560, 203] || (CO.REG:618057-T) [227, 212, 412, 240] || LOT 1851-A & 1851-B, JALAN KPB 6, [98, 252, 532, 281] || KAWASAN PERINDUSTRIAN BALAKONG, [60, 290, 574, 321] || 43300 SERI KEMBANGAN, [156, 332, 474, 359] || SELANGOR DARUL EHSAN [140, 369, 487, 396] || GST REG NO: 000854497280 [140, 410, 485, 439] || TAX INVOICE [220, 482, 410, 513] || CASHIER: C1 [36, 518, 178, 545] || DATE: 15-05-18 [397, 518, 572, 547] || TIME: 16:21:49 [400, 552, 567, 584] || RECEIPT NO: 0107297 [34, 554, 252, 582] || -------------------------------------------------- [34, 621, 567, 656] || 9054045 [30, 664, 126, 693] || CAR SHAMPOO [136, 664, 311, 693] || 1 [400, 664, 420, 693] || 7.50 [516, 664, 572, 693] || SR [597, 664, 625, 693] || -------------------------------------------------- [24, 703, 564, 735] || TOTAL: RM 7.50 [156, 742, 570, 786] || CASH: RM 10.00 [347, 793, 570, 826] || CHANGE: RM 2.50 [327, 835, 570, 865] || GOODS SOLD ARE NOT REFUNDABLE / RETURNABLE [30, 999, 556, 1022] || THANK YOU, PLEASE COME AGAIN [110, 1039, 464, 1064]"

evaluate_custom_invoice(unseen_mr_diy, "MR DIY Hardware Receipt")


Evaluating: MR DIY Hardware Receipt...
Thought: 1. Merchant: The merchant name is found at the very top of the receipt, so it's "MR D.I.Y. (M) SDN BHD".
  * Merchant: "MR D.I.Y. (M) SDN BHD"
 2. Date: The date is found immediately after the time, so it's "15-05-18".
  * Date: "15-05-18"
 3. Address: The address is spread across several lines but can be extracted most accurately as "LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR DARUL EHSAN".
  * Address: "LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR DARUL EHSAN"
 4. Total: The total is clearly stated as "RM 7.50".
  * Total: "7.50"

JSON:
```json

JSON:
{
  "merchant": "MR D.I.Y. (M) SDN BHD",
  "date": "15-05-18",
  "address": "LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR DARUL EHSAN",
  "total": "7.50"
}


In [12]:
# HARD SHUTDOWN & UNLOAD FROM GPU

import torch
import gc

print("Initiating VRAM Hard-Shutdown for Local Models...")

# 1. Track memory before cleanup
if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated Before: {vram_before:.2f} GB")

# 2. List of every heavy object that might be trapped in memory
heavy_objects = [
    'model', 'tokenizer', 'trainer', 'bertscore', 'rouge', 
    'dataset', 'train_dataset', 'eval_dataset', 'split_dataset',
    'inputs', 'outputs'
]

# 3. Delete them dynamically if they exist
for obj in heavy_objects:
    if obj in globals():
        print(f"Unloading '{obj}' from memory...")
        del globals()[obj]

# 4. Force aggressive Garbage Collection (Run twice to clear circular references)
gc.collect()
gc.collect()

# 5. Flush the GPU Cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # Clears memory shared between backend processes
    torch.cuda.synchronize()  # Waits for all GPU operations to completely finish
    
    # Track memory after cleanup
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated After:  {vram_after:.2f} GB")

print("\nGPU Memory Cleared. Your VRAM is now completely empty!")

Initiating VRAM Hard-Shutdown for Local Models...
VRAM Allocated Before: 3.32 GB
Unloading 'model' from memory...
Unloading 'tokenizer' from memory...
Unloading 'bertscore' from memory...
Unloading 'inputs' from memory...
Unloading 'outputs' from memory...
VRAM Allocated After:  2.32 GB

GPU Memory Cleared. Your VRAM is now completely empty!
